# Hands-on Exercise 2: Extract innovation-culture triples with RAG

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/helenlu-vbs/NLP_LLM_for_Finance_and-Accounting_Research-Sheffield-/blob/main/006_pseudo_analyst_report_rag_triples_teaching.ipynb)

This notebook is a small, classroom-friendly demonstration of how RAG can help extract cause-effect triples about corporate culture from a pseudo analyst report.

The example is inspired by Li, Mai, Shen, Yang & Zhang (2026), but it is **not a replication**. The goal is to make the mechanics visible in a 30-45 minute teaching session.


## Learning goals

By the end of this exercise, students should understand:

- why a focal culture segment may not contain enough evidence to extract causes and effects;
- how keyword search can identify a focal firm and focal culture segment;
- how the paper's RAG step retrieves additional context once a segment says more context is needed;
- how semantic similarity ranks additional segments from the same report;
- why the same CoT extraction prompt is rerun after RAG context is added;
- why this teaching notebook stops at original triples, while the paper later uses clustering and human input for canonicalization;
- why extracted triples still need human audit before becoming research variables.


## Protocol used in this demo

The paper's analyst-report workflow can be simplified as:

1. **Identify culture-related focal text.** In this demo, we use keyword search to identify the focal firm and an explicit innovation/adaptability culture segment.
2. **First-pass extraction.** Gemini reads only the focal segment using a Chain-of-Thought-style structured prompt.
3. **Trigger RAG if needed.** If either causes or outcomes are missing, or if any key field says `I need more context`, the segment needs RAG.
4. **Retrieve context once.** For that focal segment, retrieve context from the same pseudo report using cosine similarity between embeddings. Keep up to five most similar segments after filtering by the 75th percentile of culture-relevance probabilities. Also include immediately preceding and following segments.
5. **Assemble augmented context.** Order selected segments by report order and insert `[...]` between non-adjacent segments.
6. **Rerun the same prompt once.** Gemini receives the focal segment plus retrieved context and attempts relation extraction again. If no cause or effect can be confidently extracted even with context, it returns an empty array for that field.
7. **Stop at original triples.** Gemini returns original cause-effect triples in JSON, with a tone for each triple. We do not canonicalize them in this teaching notebook.

This matches the RAG logic in Li et al. more closely than an iterative hand-picked context loop.


In [ ]:
# Quiet package installation for Colab / fresh environments.
import subprocess
import sys

packages = [
    "pandas", "numpy", "scikit-learn", "sentence-transformers",
    "google-genai", "json-repair", "tqdm",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)


In [ ]:
import json
import os
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
from google import genai
from google.genai import types
from json_repair import repair_json
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    userdata = None
    IN_COLAB = False

MODEL_NAME = "gemini-2.5-flash-lite"
TEMPERATURE = 0
TOP_K_RAG_SEGMENTS = 5
CULTURE_PROB_PERCENTILE = 75
MAX_CONTEXT_CHARS = 12000
OUTPUT_DIR = Path("/content/outputs") if IN_COLAB else Path(r"C:\Users\Helen\Dropbox\Sheffield_NLP_teaching\outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def get_gemini_api_key():
    if userdata is not None:
        try:
            key = userdata.get("Gemini_API_Key")
            if key:
                return key
        except Exception:
            pass
    return os.getenv("Gemini_API_Key")


api_key = get_gemini_api_key()
client = genai.Client(api_key=api_key) if api_key else None

print(f"Running in Colab: {IN_COLAB}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Gemini_API_Key set: {bool(api_key)}")
print(f"Gemini model: {MODEL_NAME}")


## 1. Create a pseudo segmented analyst report

The pseudo report contains multiple firms and multiple report segments. The focal segment is **not hard-coded** later. Instead, we will identify it by keyword search.

Each segment has:

- `firm_name`
- `report_id`
- `segment_id`
- `segment_order`
- `text`
- `culture_probability`

The `culture_probability` is a teaching stand-in for the paper's BERT-based culture-relevance probability. In a real implementation, this would come from a trained model.


In [ ]:
segments = [
    {
        "firm_name": "NovaCloud",
        "report_id": "NovaCloud_Analyst_Report_2026",
        "segment_id": "A01",
        "segment_order": 1,
        "text": "NovaCloud reported 18% revenue growth, above our 14% estimate, driven by stronger enterprise demand and better cloud utilisation.",
        "culture_probability": 0.10,
    },
    {
        "firm_name": "NovaCloud",
        "report_id": "NovaCloud_Analyst_Report_2026",
        "segment_id": "A02",
        "segment_order": 2,
        "text": "Since the new CEO took over, NovaCloud shifted from a centralized AI product-cycle process to small autonomous product squads with direct customer feedback loops, a design intended to spread experimentation beyond the founder-led engineering team.",
        "culture_probability": 0.92,
    },
    {
        "firm_name": "NovaCloud",
        "report_id": "NovaCloud_Analyst_Report_2026",
        "segment_id": "A03",
        "segment_order": 3,
        "text": "Management gives senior engineers protected time to test experimental AI ideas and requires teams to share post-mortems when product projects fail, reinforcing an organization-wide habit of learning from experiments.",
        "culture_probability": 0.90,
    },
    {
        "firm_name": "NovaCloud",
        "report_id": "NovaCloud_Analyst_Report_2026",
        "segment_id": "A04",
        "segment_order": 4,
        "text": "We view NovaCloud's ability to sustain its AI product cycle as tied to whether this experimentation culture can scale beyond the founder-led engineering team.",
        "culture_probability": 0.97,
    },
    {
        "firm_name": "NovaCloud",
        "report_id": "NovaCloud_Analyst_Report_2026",
        "segment_id": "A05",
        "segment_order": 5,
        "text": "Where product squads have adopted this experimentation culture and cadence, release cycles have shortened from quarterly to monthly and net revenue retention improved from 116% to 124% as customers adopted more AI modules.",
        "culture_probability": 0.89,
    },
    {
        "firm_name": "NovaCloud",
        "report_id": "NovaCloud_Analyst_Report_2026",
        "segment_id": "A06",
        "segment_order": 6,
        "text": "Our price target is based on 9.5x next-twelve-month revenue, a discount to high-growth software peers.",
        "culture_probability": 0.08,
    },
    {
        "firm_name": "RetailHub",
        "report_id": "RetailHub_Analyst_Report_2026",
        "segment_id": "B01",
        "segment_order": 1,
        "text": "RetailHub continues to face pressure from discount peers, and we expect gross margin to remain below management's long-run target.",
        "culture_probability": 0.12,
    },
    {
        "firm_name": "RetailHub",
        "report_id": "RetailHub_Analyst_Report_2026",
        "segment_id": "B02",
        "segment_order": 2,
        "text": "Management describes the company culture as service-oriented, with store managers rewarded for customer satisfaction and employee retention.",
        "culture_probability": 0.88,
    },
]

df = pd.DataFrame(segments)
display(df)


## 2. Identify the focal firm and focal culture segment by keyword search

For teaching, we search for a focal firm keyword and innovation/adaptability culture phrases. This replaces the old hard-coded `focal_id = "A04"`.

The search has two pieces:

1. `FOCAL_FIRM_KEYWORDS`: which firm/report should be used for the exercise?
2. `FOCAL_CULTURE_KEYWORDS`: which segments explicitly mention an innovation/adaptability culture concept?

The highest culture-probability matching segment becomes the focal segment.


In [ ]:
FOCAL_FIRM_KEYWORDS = ["novacloud"]
FOCAL_CULTURE_KEYWORDS = [
    "experimentation culture",
    "innovative culture",
    "innovation culture",
    "entrepreneurial culture",
    "adaptive culture",
    "agile culture",
    "technology-driven culture",
    "data-driven culture",
    "continuous improvement culture",
]


def contains_any(text: str, keywords: list[str]) -> bool:
    text = str(text).lower()
    return any(re.search(r"\b" + re.escape(k.lower()) + r"\b", text) for k in keywords)


search_df = df.copy()
search_df["firm_keyword_hit"] = search_df["firm_name"].map(lambda x: contains_any(x, FOCAL_FIRM_KEYWORDS))
search_df["culture_keyword_hit"] = search_df["text"].map(lambda x: contains_any(x, FOCAL_CULTURE_KEYWORDS))

focal_candidates = search_df[search_df["firm_keyword_hit"] & search_df["culture_keyword_hit"]].copy()
if focal_candidates.empty:
    raise ValueError("No focal segment found. Try broader FOCAL_FIRM_KEYWORDS or FOCAL_CULTURE_KEYWORDS.")

focal_candidates = focal_candidates.sort_values(
    ["culture_probability", "segment_order"], ascending=[False, True]
).reset_index(drop=True)
focal_row = focal_candidates.iloc[0]
focal_id = focal_row["segment_id"]
focal_report_id = focal_row["report_id"]

print("Focal candidates from keyword search:")
display(focal_candidates[["firm_name", "report_id", "segment_id", "segment_order", "culture_probability", "text"]])
print(f"Selected focal segment: {focal_id} from {focal_report_id}")


## 3. Li-style Gemini extraction prompt

The prompt below follows the logic of Table 1 in the paper more closely. It asks the model to:

1. summarize the specific corporate culture;
2. classify the culture type;
3. decide whether there is detailed causal analysis or whether more context is needed;
4. identify causes;
5. identify outcomes;
6. determine tone;
7. extract original causal graph triples.

The same prompt is used for the first pass and the RAG-augmented pass.


In [ ]:
LI_STYLE_EXTRACTION_PROMPT = """
As an expert specializing in corporate culture and causal reasoning, your task is to analyze segments from analyst reports about corporate culture. Your goal is to extract and interpret information about a company's corporate culture and identify cause-effect relationships. Present your findings in a structured JSON format. Let's think step by step.

Step-by-Step Instructions:

1. Summarize the Corporate Culture (Q1):
   - Task: Determine the specific corporate culture being discussed in the segment.
   - Action: Summarize it in a short phrase starting with an adjective. If corporate culture is not explicitly mentioned, infer it from the context.
   - Note: Avoid generic adjectives such as strong/weak or positive/negative culture. If more context from the report is needed for the analysis, output "I need more context" in the relevant JSON field.

2. Classify the Corporate Culture (Q2):
   - Task: Categorize the identified corporate culture into one of the following six types:
     * Collaboration and People-Focused: Focusing on collaboration, cooperation, teamwork, supportive behavior, low levels of conflict, community, communication within an organization, employee well-being, employee equity sharing and compensation, diversity, inclusion, empowerment, or talent.
     * Customer-Oriented: Focusing on sales, customer, customer service, listening to the customer, customer retention, customer experience, customer satisfaction, user experience, client service, being brand-driven, quality of product, quality of service, quality of solution, or taking pride in service.
     * Innovation and Adaptability: Focusing on innovation, creativity, technology, entrepreneurship, adaptability, transformation, flexibility, agility, willingness to experiment, beyond tradition, disruption, fast-moving, quick to take advantage of opportunities, resilience to change, or taking initiative.
     * Integrity and Risk Management: Focusing on integrity, high ethical standards, being honest, being transparent, accountability, do the right thing, fair practices, being trustworthy, risk management, risk control, compliance, discipline, or financial prudence.
     * Performance-Oriented: Focusing on high expectations for performance, sales growth, achievement, competitiveness, results, hard work, efficiency, productivity, consistency in executing tasks, setting clear goals, following best practices, striving for operational excellence, or exceeding benchmarks.
     * Miscellaneous: Nonspecific corporate culture, or corporate culture that does not easily fit into the above types.

3. Identify Detailed Causal Analysis (Q3):
   - Task: Determine if the segment contains a detailed causal analysis of corporate culture.
   - Criteria: Look for explicit causal reasoning statements with trigger words like affect, cause, influence, lead to, result in, fosters, driven by.
   - Action: Return "YES", "NO", or "I need more context".
   - If more context is needed, output "I need more context" in this field.

4. Identify Causes of Corporate Culture (Q4):
   - Task: If detailed causal analysis exists, identify explicitly mentioned events or factors that have shaped, changed, or will change the corporate culture.
   - Action: List the important causes or return an empty array [] if not applicable.
   - Note: Do not list other corporate culture types as causes. Focus on specific people, systems, events, practices, or implicit/indirect causes.

5. Identify Outcomes from Corporate Culture (Q5):
   - Task: If detailed causal analysis exists, identify explicitly mentioned past, present, or future outcomes or impacts of the corporate culture on the company.
   - Action: List the important outcomes or return an empty array [] if not applicable.
   - Note: Do not list other corporate culture types as outcomes. Focus on specific results, business outcomes, or tangible impacts. Do not treat vague or conditional statements such as "ability is tied to whether the culture can scale" as an outcome. Such statements may justify requesting more context, but they should not be listed as outcomes and should not appear in causal_graph_triples.

6. Determine the Tone (Q6):
   - Task: Assess the tone of the discussion about corporate culture.
   - Options: "positive", "negative", "neutral".
   - Note: If the tone is unclear, mark it as "neutral".

7. Extract Causal Graph Triples (Q7):
   - Task: Based on the answers from Q1, Q4, and Q5, extract causal graph triples related to that specific corporate culture.
   - Format for each triple: ["entity_1", "relation", "entity_2"]
   - Tone: For each triple, assign a triple-level tone: "positive", "negative", or "neutral". The tone should reflect whether the causal relation is described as beneficial, harmful, or descriptive/unclear.
   - Explanation: Provide a brief reason citing specific words or phrases.
   - Criteria:
     * "entity_1" or "entity_2" must be the specific corporate culture identified in Q1.
     * "relation" should be a clear, simple verb phrase showing the cause-effect direction.
     * The other entity should be either a cause or an outcome for the specific corporate culture, not another corporate culture.
   - Avoid both entities being corporate culture.
   - Do not create triples from vague conditional statements such as "ability is tied to whether the culture can scale." Use those statements only as evidence that more context may be needed.

Return strict JSON only using this structure:
{
  "all_results": [
    {
      "input_id": "XXXX",
      "identified_corporate_culture": "adjective + specific corporate culture" or "I need more context",
      "corporate_culture_type": "one of the six types" or "I need more context",
      "detailed_causal_analysis": "YES" or "NO" or "I need more context",
      "causes_of_culture": ["cause_1", "..."] or [],
      "outcomes_from_culture": ["outcome_1", "..."] or [],
      "tone": "positive" or "negative" or "neutral",
      "causal_graph_triples": [
        {
          "triple": ["entity_1", "relation", "entity_2"],
          "tone": "positive" or "negative" or "neutral",
          "explanation": "Reason for identifying the causal relation, citing specific words or phrases."
        }
      ] or []
    }
  ]
}

Input segments:
{input_json}
"""

print(LI_STYLE_EXTRACTION_PROMPT[:1600])


## 4. Gemini helper functions

These helper functions call Gemini Flash Lite using the current `google-genai` package. The notebook expects the API key to be stored as a Colab secret named exactly `Gemini_API_Key`.

The parser accepts the paper-style `all_results` JSON output and returns the first result because this teaching notebook processes one focal segment at a time.


In [ ]:
def parse_json_response(text: str) -> dict:
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        parsed = json.loads(repair_json(text))

    if isinstance(parsed, dict) and "all_results" in parsed:
        results = parsed["all_results"]
        if not results:
            raise ValueError("Gemini returned an empty all_results list")
        parsed = results[0]

    if not isinstance(parsed, dict):
        raise ValueError(f"Expected JSON object, got {type(parsed)}")
    return parsed


def call_gemini_json(prompt: str, max_retries: int = 3) -> dict:
    if client is None:
        raise ValueError(
            "Set Gemini_API_Key in Colab Secrets, or set an environment variable named Gemini_API_Key."
        )
    last_error = None
    for attempt in range(1, max_retries + 1):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=TEMPERATURE,
                    response_mime_type="application/json",
                ),
            )
            return parse_json_response(response.text)
        except Exception as exc:
            last_error = exc
            wait_seconds = 5 * attempt
            print(f"Gemini call failed on attempt {attempt}/{max_retries}: {exc}")
            time.sleep(wait_seconds)
    raise RuntimeError("Gemini failed after retries") from last_error


def make_input_json(focal_row, context_rows=None) -> str:
    text_parts = [f"FOCAL {focal_row['segment_id']}: {focal_row['text']}"]

    if context_rows is not None and len(context_rows) > 0:
        text_parts.append("ADDITIONAL CONTEXT FROM SAME REPORT:")
        prev_order = None
        for _, row in context_rows.sort_values("segment_order").iterrows():
            order = int(row["segment_order"])
            if prev_order is not None and order > prev_order + 1:
                text_parts.append("[...]")
            text_parts.append(
                f"{row['segment_id']} | similarity={row['similarity_to_focal']:.3f} | "
                f"culture_probability={row['culture_probability']:.2f}: {row['text']}"
            )
            prev_order = order

    payload = [{"input_id": str(focal_row["segment_id"]), "segment": "\n".join(text_parts)}]
    return json.dumps(payload, ensure_ascii=False)


def extract_with_gemini(focal_row, context_rows=None) -> dict:
    input_json = make_input_json(focal_row, context_rows)
    prompt = LI_STYLE_EXTRACTION_PROMPT.replace("{input_json}", input_json)
    result = call_gemini_json(prompt)
    result["input_json_used"] = input_json
    return result


def needs_rag(result: dict) -> bool:
    need_text = "i need more context"
    key_fields = [
        str(result.get("identified_corporate_culture", "")).lower(),
        str(result.get("corporate_culture_type", "")).lower(),
        str(result.get("detailed_causal_analysis", "")).lower(),
    ]
    if any(need_text in field for field in key_fields):
        return True
    if len(result.get("causes_of_culture", []) or []) == 0:
        return True
    if len(result.get("outcomes_from_culture", []) or []) == 0:
        return True
    return False


## 5. First pass: focal segment only

Gemini first sees only the focal segment. If it can identify the culture but lacks sufficient cause/effect evidence, it should request more context.


In [ ]:
round1 = extract_with_gemini(focal_row)
print(json.dumps({k: v for k, v in round1.items() if k != "input_json_used"}, indent=2))


## 6. Retrieve RAG context once using semantic similarity

This is the paper-style RAG step. If the focal segment needs more context, we retrieve additional context from the same report once:

1. compute embeddings for all same-report segments;
2. compute cosine similarity to the focal segment;
3. filter candidate context segments by the 75th percentile of culture-relevance probabilities;
4. keep up to five most similar filtered segments;
5. also include the immediately preceding and following segments;
6. order the final context by report order and use `[...]` between non-adjacent segments.

This is different from a hand-picked context example: the retrieval rule determines which text Gemini sees.


In [ ]:
def build_rag_context_once(df, focal_row, top_k=TOP_K_RAG_SEGMENTS, percentile=CULTURE_PROB_PERCENTILE):
    same_report = df[df["report_id"].eq(focal_row["report_id"])].copy().reset_index(drop=True)
    texts = same_report["text"].tolist()

    embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
    embeddings = embedder.encode(texts, show_progress_bar=False, batch_size=16)
    focal_pos = same_report.index[same_report["segment_id"].eq(focal_row["segment_id"])][0]
    sims = cosine_similarity([embeddings[focal_pos]], embeddings)[0]

    same_report["similarity_to_focal"] = sims
    prob_cutoff = np.percentile(same_report["culture_probability"], percentile)

    non_focal = same_report[same_report["segment_id"].ne(focal_row["segment_id"])].copy()
    filtered = non_focal[non_focal["culture_probability"] >= prob_cutoff].copy()
    top_similar = (
        filtered.sort_values("similarity_to_focal", ascending=False)
        .head(top_k)
        .assign(retrieval_reason="top_semantic_after_culture_probability_filter")
    )

    focal_order = int(focal_row["segment_order"])
    adjacent = non_focal[non_focal["segment_order"].isin([focal_order - 1, focal_order + 1])].copy()
    adjacent["retrieval_reason"] = "adjacent_segment"

    selected = pd.concat([top_similar, adjacent], ignore_index=True)
    if selected.empty:
        selected = top_similar.copy()

    selected = (
        selected.groupby(["firm_name", "report_id", "segment_id", "segment_order", "text", "culture_probability"], as_index=False)
        .agg({
            "similarity_to_focal": "max",
            "retrieval_reason": lambda x: " + ".join(sorted(set(x))),
        })
        .sort_values(["segment_order", "similarity_to_focal"], ascending=[True, False])
        .reset_index(drop=True)
    )
    selected["retrieval_rank_by_similarity"] = selected["similarity_to_focal"].rank(method="first", ascending=False).astype(int)

    # Classroom token-budget simplification: cap by characters, dropping least similar segments first.
    def context_length(frame):
        return int(frame["text"].str.len().sum())

    while len(selected) > 0 and context_length(selected) > MAX_CONTEXT_CHARS:
        drop_idx = selected["similarity_to_focal"].idxmin()
        selected = selected.drop(index=drop_idx).reset_index(drop=True)

    ranking = non_focal.sort_values("similarity_to_focal", ascending=False).reset_index(drop=True)
    ranking["retrieval_rank_by_similarity"] = np.arange(1, len(ranking) + 1)
    ranking["passes_culture_probability_filter"] = ranking["culture_probability"] >= prob_cutoff

    return selected, ranking, prob_cutoff


rag_context, retrieval_ranking, prob_cutoff = build_rag_context_once(df, focal_row)
print(f"75th percentile culture-probability cutoff: {prob_cutoff:.3f}")
print("Full same-report semantic ranking:")
display(retrieval_ranking[["retrieval_rank_by_similarity", "segment_id", "segment_order", "similarity_to_focal", "culture_probability", "passes_culture_probability_filter", "text"]])
print("Selected RAG context:")
display(rag_context[["segment_id", "segment_order", "similarity_to_focal", "culture_probability", "retrieval_reason", "text"]])


## 7. Rerun the same extraction prompt with RAG context

If either cause or effect is missing from the focal-only result, we rerun the **same** CoT extraction prompt with focal segment plus the retrieved RAG context.

Teaching note: in this pseudo report, `A02` should provide cause evidence because it describes CEO-led autonomous product squads and feedback loops. It is not sufficient by itself for both cause and effect. Concrete effect evidence should come from text such as `A05`, which discusses shorter release cycles and improved net revenue retention.


In [ ]:
if needs_rag(round1):
    print("First pass needs more context, so we rerun extraction with paper-style RAG context.")
    final_context_segments = rag_context.copy()
    final_result = extract_with_gemini(focal_row, final_context_segments)
    rag_decision = {
        "first_pass_needed_rag": True,
        "reason": "Either causes or outcomes were missing, or a field requested more context.",
        "selected_context_segment_ids": final_context_segments["segment_id"].tolist(),
        "culture_probability_cutoff": float(prob_cutoff),
    }
else:
    print("First pass had causes and outcomes; no RAG context needed.")
    final_context_segments = rag_context.head(0).copy()
    final_result = round1
    rag_decision = {
        "first_pass_needed_rag": False,
        "reason": "Focal segment already had causes and outcomes.",
        "selected_context_segment_ids": [],
        "culture_probability_cutoff": float(prob_cutoff),
    }

print("\nRAG decision:")
print(json.dumps(rag_decision, indent=2))
print("\nFinal extraction:")
print(json.dumps({k: v for k, v in final_result.items() if k != "input_json_used"}, indent=2))


## 8. Show Gemini's original triples

Each row is one extracted causal triple returned by Gemini. We intentionally do **not** canonicalize causes or effects here, because the paper performs canonicalization later using clustering and human input. For teaching, we stop at the original triples and audit their evidence.


In [ ]:
triple_rows = []
for t in final_result.get("causal_graph_triples", []):
    triple = t.get("triple", [])
    if isinstance(triple, list) and len(triple) == 3:
        source, relation, target = [str(x) for x in triple]
    else:
        source = str(t.get("source_entity", ""))
        relation = str(t.get("relation", ""))
        target = str(t.get("target_entity", ""))

    triple_rows.append({
        "focal_firm": focal_row["firm_name"],
        "focal_segment": focal_row["segment_id"],
        "source_entity_original": source,
        "relation_original": relation,
        "target_entity_original": target,
        "original_triple": f"{source} -> {relation} -> {target}",
        "explanation": t.get("explanation", ""),
        "tone": t.get("tone", final_result.get("tone", "")),
        "culture_type": final_result.get("corporate_culture_type", ""),
    })

triples_df = pd.DataFrame(triple_rows)
display(triples_df)


## 9. Show final evidence

Students should audit whether Gemini's extracted triples are supported by the focal segment and retrieved context.


In [ ]:
if not triples_df.empty:
    display(triples_df[["original_triple", "tone", "explanation"]])
else:
    print("No triples extracted. If causes or outcomes are still missing after RAG, the paper-style output is an empty triple array.")


## 10. Save teaching outputs

The outputs let students inspect each stage: focal candidates, semantic retrieval ranking, RAG iterations, and final Gemini triples.


In [ ]:
df.to_csv(OUTPUT_DIR / "006_pseudo_report_segments.csv", index=False)
focal_candidates.to_csv(OUTPUT_DIR / "006_focal_keyword_candidates.csv", index=False)
retrieval_ranking.to_csv(OUTPUT_DIR / "006_rag_semantic_retrieval_ranking.csv", index=False)
final_context_segments.to_csv(OUTPUT_DIR / "006_rag_selected_context.csv", index=False)
triples_df.to_csv(OUTPUT_DIR / "006_pseudo_report_rag_triples.csv", index=False)

with open(OUTPUT_DIR / "006_rag_decision.json", "w", encoding="utf-8") as f:
    json.dump(rag_decision, f, indent=2, ensure_ascii=False)
with open(OUTPUT_DIR / "006_final_gemini_extraction.json", "w", encoding="utf-8") as f:
    json.dump({k: v for k, v in final_result.items() if k != "input_json_used"}, f, indent=2, ensure_ascii=False)

print("Saved outputs to:", OUTPUT_DIR)
for path in sorted(OUTPUT_DIR.glob("006_*")):
    print(path.name)


## 11. Class discussion

1. How was the focal segment identified? What keyword choices mattered?
2. Which segments passed the 75th percentile culture-probability filter?
3. Which selected context segments came from semantic similarity, and which came from adjacency?
4. Did the RAG context provide missing causes, outcomes, or both?
5. Did Gemini infer anything not supported by the text?
6. Why does this notebook stop at original triples rather than canonicalizing causes and effects?
7. How is this demo similar to and different from Li et al. (2026)?

## What we learned

- RAG context should be retrieved by a rule, not hand-picked after seeing the answer.
- The paper triggers RAG when the focal segment lacks enough evidence for cause/effect extraction.
- Semantic retrieval gives an auditable ranking from closest to farthest context.
- Adjacent segments are added because surrounding text often clarifies context.
- Gemini can extract original triples, but canonical categories require clustering and human input before research use.
